# UEMR 0-shot trên HT-Step (Athena sạch + Omni pt / ft)

Theo `docs/HTSTEP_0SHOT_UEMR.md`. Mọi tham số, danh sách video, tiêu chí: `configs/htstep_0shot_protocol.json` (khoá, commit trước lượt
này; ô 2 kiểm sha256). **Không train, không chọn tham số trên HT-Step, val chấm một lần.** Athena sạch cắt đoạn **một lần** (lưu kèm hash);
mỗi lượt Omni kiểm hash đó.

| Ô | Việc | Cần |
|---|---|---|
| 0 | cấu hình | |
| 1 | Drive, code Fushion, thư viện, unit test, sha256 Athena sạch | CPU |
| 2 | kiểm hash protocol | CPU |
| 3 | Bước A: kiểm kê, chọn video, lọc trùng YC2, headline trùng, đối chiếu 50 bước với repo HT-Step | CPU |
| 4 | Bước A: lấy mp4 val theo tar (resume), thời lượng mp4, câu hỏi | CPU, ~9 GB tải tạm |
| 5 | Bước A: đặc trưng (`htstep-feats/feats`, trích bổ sung nếu thiếu), kiểm định dạng so với YC2 | CPU (GPU nếu phải trích) |
| 6 | Bước B: kiểm tái lập Athena trên 5 video YC2, cắt R6 → R8, lưu hash | GPU bất kỳ (T4 được) |
| 7 | Bước B.3: chất lượng cắt so với GT | CPU |
| 8 | Bước C: R0, R2, R3, R7; đếm clip; ước lượng thời gian | CPU |
| 9 | Bước D: Omni encode từng tag (kiểm tái lập 5 clip YC2, đo 20 clip, resume) | **A100 hoặc L4** (≥ 22 GB) |
| 10 | Bước E: chấm từng tag (strict + multi, mức 1 và 2) | CPU |
| 11 | tiêu chí mục 6 → `criteria.json` | CPU |
| 12 | Bước F: so pt với ft (cần đủ hai tag) | CPU |
| 13 | xuất kết quả, tóm tắt | CPU |

**Cách chạy:** Runtime **A100** (hoặc L4), **Run all**. Ngắt giữa chừng: Run all lại; mọi bước có resume (mp4 theo tar, đoạn có hash,
encode ghi store theo lô 200 clip và bỏ qua clip đã có). Không đủ thời gian cho cả hai Omni: chạy lại với `OMNI_TAGS` còn lại; ô 11–12 chỉ
chạy phần đã đủ dữ liệu và in rõ thiếu gì. Mỗi ô in `CHECK … OK/FAIL`; có `FAIL` thì ô dừng: gửi log, không sửa tay.
Gửi lại log **mọi ô**.

In [ ]:
# ===== 0. Cấu hình =====
FUSHION_REF = 'c1ac3552b8418d9d42035fb7a1078689786571b1'   # commit trên nhánh htstep chứa configs/htstep_0shot_protocol.json
PROTO_SHA16 = 'c657a44bcd1d762b'   # sha256[:16] của configs/htstep_0shot_protocol.json tại FUSHION_REF
OMNI_TAGS   = ['pt', 'ft']   # một phiên không đủ: chạy lại với tag còn lại
SEED        = 0
RUN_ATHENA  = True           # ô 6-8 (đã có đoạn + hash trên Drive thì tự đọc lại, không cắt lại)
RUN_OMNI    = True           # ô 9
RUN_EVAL    = True           # ô 10-13
FLAGS = {k: v for k, v in dict(globals()).items() if k.startswith('RUN_') or k in ('FUSHION_REF', 'PROTO_SHA16', 'OMNI_TAGS', 'SEED')}

In [ ]:
# ===== 1. Drive, code Fushion, thư viện, unit test, sha256 Athena sạch =====
from google.colab import drive
drive.mount('/content/drive')
import os, sys, json, glob, shutil, subprocess, time, hashlib, platform, random
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor
os.environ['HF_XET_CHUNK_CACHE_SIZE_BYTES'] = '0'      # không giữ bản sao tar trong cache xet (đĩa)
os.environ.pop('HF_XET_HIGH_PERFORMANCE', None)         # chế độ này đệm nhiều GB trong RAM
T_START = time.time()
FUSHION_GIT = 'https://github.com/chien24/Fushion_UEMR.git'
FUSHION     = '/content/code/Fushion_UEMR'
os.makedirs('/content/code', exist_ok=True)
shutil.rmtree(FUSHION, ignore_errors=True)
subprocess.run(['git', 'clone', '-q', FUSHION_GIT, FUSHION], check=True)
git = lambda *a: subprocess.run(['git', '-C', FUSHION, *a], capture_output=True, text=True).stdout.strip()
r = subprocess.run(['git', '-C', FUSHION, 'checkout', '-q', FUSHION_REF], capture_output=True, text=True)
assert r.returncode == 0, f'không checkout được FUSHION_REF={FUSHION_REF}: {r.stderr} (đã push nhánh htstep chưa?)'
COMMIT = git('log', '--oneline', '-1')
print('code:', git('rev-parse', 'HEAD'), '|', COMMIT)
if FUSHION not in sys.path:
    sys.path.insert(0, FUSHION)
os.environ['PYTHONPATH'] = FUSHION + ':' + os.environ.get('PYTHONPATH', '')
os.chdir(FUSHION)

def sh(cmd, check=True):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    out = (r.stdout + r.stderr)[-3000:]
    if out.strip():
        print(out)
    if check and r.returncode:
        raise RuntimeError(f'lệnh lỗi ({r.returncode}): {cmd[:200]}')
    return r

def check(name, ok, detail=''):
    print(f"CHECK {name}: {'OK' if ok else 'FAIL'}" + (f' | {detail}' if detail else ''))
    assert ok, f'CHECK {name} FAIL: {detail}'

sh(f'pip install -q --only-binary numpy -r {FUSHION}/requirements_uniav.txt')
import numpy as np, pandas as pd, torch
from IPython.display import display
check('numpy >= 2 (checkpoint Athena)', int(np.__version__.split('.')[0]) >= 2,
      f'numpy {np.__version__}: Runtime > Restart session rồi chạy lại từ ô 0')
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'không có'
GPU_GB = torch.cuda.get_device_properties(0).total_memory / 2**30 if torch.cuda.is_available() else 0.0
ENV = {'python': platform.python_version(), 'torch': torch.__version__, 'numpy': np.__version__, 'pandas': pd.__version__,
       'gpu': gpu, 'gpu_GB': round(GPU_GB, 1), 'code': COMMIT}
print(json.dumps(ENV, indent=1, ensure_ascii=False))
r = subprocess.run([sys.executable, '-m', 'pytest', 'tests/test_uemr_htstep.py', 'uniav_app/tests', 'omni_retrieval/tests', '-q',
                    '-p', 'no:cacheprovider'], cwd=FUSHION, capture_output=True, text=True)
print(r.stdout[-1500:])
check('unit test', r.returncode == 0, r.stdout[-300:])

def sha256(path, bs=1 << 24):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(bs), b''):
            h.update(chunk)
    return h.hexdigest()

def wj(path, obj):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    tmp = path + '.tmp'
    with open(tmp, 'w', encoding='utf-8') as f:
        json.dump(obj, f, indent=1, ensure_ascii=False, default=str)
    os.replace(tmp, path)

def rj(path, default=None):
    return json.load(open(path, encoding='utf-8')) if os.path.exists(path) else default

DRIVE    = '/content/drive/MyDrive'
BASE     = f'{DRIVE}/Colab Notebooks/code KL/Fushion/uemr'
H        = f'{BASE}/htstep_0shot'                       # mọi đầu ra của lượt này trên Drive
VID      = f'{H}/videos_val'
SEGDIR   = f'{H}/segments'
O_YC2    = f'{BASE}/omni_cache'                        # cache YC2 (chỉ đọc): kiểm tái lập
YC2_DATA = f'{DRIVE}/Colab Notebooks/code KL/Omni/data/YouCookII'
ATHENA_SHA  = '961f1a895f4b43c811d3e0cfa31448641989a0a0e39e421ed397682407643d58'
ATHENA_SRC  = f'{BASE}/athena_sach/961f1a89/athena_yc2clean.pth'
CKPT        = '/content/ckpt/athena_yc2clean.pth'
for d in (H, VID, SEGDIR):
    os.makedirs(d, exist_ok=True)
os.makedirs('/content/ckpt', exist_ok=True)
if not (os.path.exists(CKPT) and sha256(CKPT) == ATHENA_SHA):
    shutil.copy2(ATHENA_SRC, CKPT)
check('sha256 Athena sạch', sha256(CKPT) == ATHENA_SHA, CKPT)

In [ ]:
# ===== 2. Kiểm hash protocol =====
PROTO_PATH = f'{FUSHION}/configs/htstep_0shot_protocol.json'
PH = sha256(PROTO_PATH)
check('sha256 protocol', PH[:16] == PROTO_SHA16, f'tại {FUSHION_REF}: {PH[:16]}, notebook ghi {PROTO_SHA16}')
PROTO_COMMIT = git('log', '-1', '--format=%h %ad', '--date=short', '--', 'configs/htstep_0shot_protocol.json')
check('protocol đã commit', bool(PROTO_COMMIT), PROTO_COMMIT)
PR = json.load(open(PROTO_PATH, encoding='utf-8'))
for k, s in PR['sources'].items():
    check(f'nguồn {k} không đổi', sha256(f"{FUSHION}/{s['path']}") == s['sha256'], s['path'])
check('Athena trong protocol', PR['models']['athena']['sha256'] == ATHENA_SHA)
PP, PAR, EV, CR = PR['params']['uniav_postprocess'], PR['params'], PR['eval'], PR['criteria']
THETA = float(PAR['uniav_theta'])
NB, BSEED = EV['bootstrap']['n_boot'], EV['bootstrap']['seed']
RES = f'{H}/results/{PROTO_SHA16[:8]}'
os.makedirs(RES, exist_ok=True)
print(json.dumps({'protocol': PH[:16], 'commit': PROTO_COMMIT, 'θ': THETA, 'postprocess': PP, 'L_min': PAR['L_min_s'],
                  'L_fill': PAR['L_fill_s'], 'R2 L': PAR['R2_L_s'], 'bootstrap': EV['bootstrap'], 'kết quả': RES}, ensure_ascii=False))
MAN_PATH = f'{H}/run_manifest.json'
MAN = rj(MAN_PATH, {'runs': []})
RUN = {'started': time.strftime('%Y-%m-%d %H:%M:%S'), 'commit': COMMIT, 'protocol_sha16': PH[:16], 'flags': FLAGS, 'env': ENV,
       'sec': {}, 'info': {}}
def done(part, t0, **info):
    RUN['sec'][part] = round(time.time() - t0, 1)
    RUN['info'][part] = info
    wj(MAN_PATH, {**MAN, 'runs': MAN['runs'] + [RUN]})
    print(f'[manifest] {part}: {RUN["sec"][part]} s')
LABEL = {'global': 'R0', 'uni_chunk': 'R2', 'uni_M': 'R3', 'uniav_pred': 'R6', 'gt': 'R7', 'r8': 'R8'}
ROWS = list(LABEL)                     # thứ tự bảng
INV = rj(f'{RES}/inventory.json', {})  # inventory.json: ô 3, 4, 5 ghi dần

In [ ]:
# ===== 3. Bước A: kiểm kê, chọn video, lọc trùng YC2, headline trùng, đối chiếu 50 bước =====
from huggingface_hub import hf_hub_download
from uemr_htstep import data as hd
t0 = time.time()
HFD, HFV = PR['data']['hf']['htstep-feats'], PR['data']['hf']['htstep-videos']
A = '/content/hts'
anno_p = hf_hub_download(HFD['repo'], HFD['anno'], repo_type='dataset', revision=HFD['revision'], local_dir=A)
man_p = hf_hub_download(HFV['repo'], HFV['manifest'], repo_type='dataset', revision=HFV['revision'], local_dir=A)
check('sha256 anno/htstep.json', sha256(anno_p) == HFD['anno_sha256'])
check('sha256 manifest.json', sha256(man_p) == HFV['manifest_sha256'])
HT = json.load(open(anno_p, encoding='utf-8'))
MANV = json.load(open(man_p, encoding='utf-8'))
TARS = hd.video_tars(MANV)
YC2_IDS = hd.yc2_video_ids(f"{FUSHION}/{PR['data']['yc2_ids']['file']}")
check('sha256 danh sách ID YC2', sha256(f"{FUSHION}/{PR['data']['yc2_ids']['file']}") == PR['data']['yc2_ids']['sha256'])
VIDEOS, COUNTS = hd.select_videos(HT, TARS, YC2_IDS)
check('danh sách video = protocol', VIDEOS == PR['data']['videos']
      and hashlib.sha256(('\n'.join(VIDEOS) + '\n').encode()).hexdigest() == PR['data']['videos_sha256'], f'{len(VIDEOS)} video')
STEPS = hd.steps(HT, VIDEOS)
check('số bước Full Match = protocol', len(STEPS) == PR['data']['steps_full_match'], str(len(STEPS)))

# đối chiếu với repo HT-Step gốc (commit khoá)
CC = PR['data']['cross_check']
HTREPO = '/content/htstep_repo'
if not os.path.isdir(f'{HTREPO}/.git'):
    subprocess.run(['git', 'clone', '-q', CC['repo'], HTREPO], check=True)
subprocess.run(['git', '-C', HTREPO, 'checkout', '-q', CC['commit']], check=True)
for f, s in CC['files'].items():
    check(f'sha256 HT-Step {f}', sha256(f'{HTREPO}/{f}') == s)
OFFICIAL = json.load(open(f'{HTREPO}/data/annotations.json', encoding='utf-8'))
TAX = pd.read_csv(f'{HTREPO}/data/taxonomy.csv')
HEADLINE_OF = dict(zip(TAX['global_step_index'].astype(int), TAX['headline']))
sample, bad = hd.cross_check(STEPS, OFFICIAL, HEADLINE_OF, CC['n'], CC['seed'])
check(f"đối chiếu {len(sample)} bước ngẫu nhiên (seed {CC['seed']}) với repo HT-Step", not bad, str([b['caption_id'] for b in bad][:5]))
check('mọi video giữ lại là val_seen trong repo HT-Step', all(OFFICIAL[v]['subset'] == 'val_seen' for v in VIDEOS))

ns = np.array([len(HT[v]['segments']) for v in VIDEOS])
sl = np.array([s['te'] - s['ts'] for s in STEPS])
dl = np.array([HT[v]['duration'] for v in VIDEOS])
pct = lambda a: {'trung vị': float(np.median(a)), 'p10': float(np.percentile(a, 10)), 'p90': float(np.percentile(a, 90)),
                 'trung bình': float(a.mean())}
HSTAT = hd.headline_stats(STEPS)
INV.update({'counts': COUNTS, 'val_seen trong video_splits.json của repo HT-Step': len(json.load(open(f'{HTREPO}/data/video_splits.json'))['val_seen']),
            'steps': len(STEPS), 'steps/video': pct(ns), 'độ dài bước (s, chưa clamp)': pct(sl),
            'thời lượng video (s, anno)': pct(dl), 'headline (mọi bước)': HSTAT,
            'cross_check': {'n': len(sample), 'lệch': len(bad)}})
print(json.dumps({k: INV[k] for k in ('counts', 'val_seen trong video_splits.json của repo HT-Step', 'steps', 'steps/video',
                                       'độ dài bước (s, chưa clamp)', 'thời lượng video (s, anno)')}, indent=1, ensure_ascii=False))
print({k: v for k, v in HSTAT.items() if k != 'top'})
display(pd.DataFrame(HSTAT['top']))
wj(f'{RES}/inventory.json', INV)
done('A_inventory', t0, counts=COUNTS)

In [ ]:
# ===== 4. Bước A: lấy mp4 val theo tar (resume), thời lượng mp4, câu hỏi =====
from uniav_app.athena_api import probe
t0 = time.time()
PROG_P = f'{H}/progress/videos.json'
PROG = rj(PROG_P, {})
PLAN = defaultdict(list)
for v in VIDEOS:
    PLAN[TARS[v]].append(v)
TMP = '/content/hv_tmp'
have = lambda v: os.path.exists(f'{VID}/{v}.mp4') and os.path.getsize(f'{VID}/{v}.mp4') > 0
print('tar cần:', {t: len(vs) for t, vs in sorted(PLAN.items())})
for i, (tar, vids) in enumerate(sorted(PLAN.items()), 1):
    todo = [v for v in vids if not have(v)]
    if not todo and PROG.get(tar, {}).get('done'):
        print(f'[{i}/{len(PLAN)}] {tar}: đã xong ({len(vids)} video)')
        continue
    size = MANV['shards'][tar]['size']
    free_local, free_drive = shutil.disk_usage('/content').free, shutil.disk_usage(DRIVE).free
    print(f'[{i}/{len(PLAN)}] {tar}: {len(todo)}/{len(vids)} video cần | tar {size / 2**30:.2f} GB | đĩa trống {free_local / 2**30:.1f} GB '
          f'| Drive trống {free_drive / 2**30:.1f} GB', flush=True)
    if free_drive < 15 * 2**30:
        print('CẢNH BÁO: Drive còn < 15 GB')
    check(f'đĩa đủ cho {tar}', free_local > 1.2 * size + 2 * 2**30, f'{free_local / 2**30:.1f} GB trống')
    path = hf_hub_download(HFV['repo'], tar, repo_type='dataset', revision=HFV['revision'], local_dir=TMP)
    found = hd.extract_members(path, {f'{v}.mp4': f'{VID}/{v}.mp4' for v in todo})
    os.remove(path)
    shutil.rmtree(TMP, ignore_errors=True)
    check(f'đã xoá {tar} và file tạm', not os.path.exists(path) and not os.path.exists(TMP))
    PROG[tar] = {'done': True, 'videos': len(vids), 'extracted': len(found), 'missing': [v for v in vids if not have(v)],
                 'time': time.strftime('%Y-%m-%d %H:%M:%S')}
    wj(PROG_P, PROG)
NO_MP4 = [v for v in VIDEOS if not have(v)]
check('mọi video của kho có mp4 trên Drive', not NO_MP4, str(NO_MP4[:5]))

# thời lượng mp4 (đọc header bằng ffmpeg, như subset_videos.json của YC2); cache theo kích thước file
VM_P = f'{H}/htstep_val_manifest.json'
VM = {r['video_id']: r for r in rj(VM_P, {'videos': []})['videos']}
def row(v):
    sz = os.path.getsize(f'{VID}/{v}.mp4')
    if v in VM and VM[v].get('size') == sz:
        return VM[v]
    return {'video_id': v, 'tar': TARS[v], 'size': sz, 'duration': round(float(probe(f'{VID}/{v}.mp4')['duration']), 3),
            'duration_anno': HT[v]['duration'], 'steps': len(HT[v]['segments'])}
with ThreadPoolExecutor(8) as pool:
    VM = {r['video_id']: r for r in pool.map(row, VIDEOS)}
wj(VM_P, {'protocol_sha16': PH[:16], 'videos': [VM[v] for v in VIDEOS]})
print('htstep_val_manifest.json sha256:', sha256(VM_P))
DUR = {v: float(VM[v]['duration']) for v in VIDEOS}
dd = np.array([DUR[v] - VM[v]['duration_anno'] for v in VIDEOS])
check('thời lượng mp4 hợp lệ', all(d > 1 for d in DUR.values()))
print(f'mp4 - anno (s): trung vị {np.median(dd):+.2f}, |lệch| > 2 s: {int((abs(dd) > 2).sum())} video')

CAPS, DROPPED = hd.queries(STEPS, DUR)
print(f'câu hỏi: {len(CAPS)} | bước bị clamp làm rỗng (loại): {len(DROPPED)} {[d["caption_id"] for d in DROPPED][:10]}')
INV.update({'mp4': {'tars': {t: len(vs) for t, vs in sorted(PLAN.items())}, 'manifest_sha256': sha256(VM_P),
                    'thời lượng mp4 (s)': pct(np.array(list(DUR.values()))), 'mp4 - anno |lệch| > 2 s': int((abs(dd) > 2).sum())},
            'queries': len(CAPS), 'dropped_by_clamp': [d['caption_id'] for d in DROPPED]})
if DROPPED:
    INV['headline (câu hỏi sau clamp)'] = hd.headline_stats(CAPS)
    print({k: v for k, v in INV['headline (câu hỏi sau clamp)'].items() if k != 'top'})
wj(f'{RES}/inventory.json', INV)
done('A_videos', t0, queries=len(CAPS), dropped=len(DROPPED))

In [ ]:
# ===== 5. Bước A: đặc trưng (htstep-feats/feats, trích bổ sung nếu thiếu), kiểm định dạng so với YC2 =====
from uniav_app.config import Config as UConfig
from uniav_app.features import get_source
t0 = time.time()
LOCAL_FEATS = '/content/htstep_feats'                   # <video_id>.npz, đọc qua FEATURE_SOURCE 'drive' của uniav_app
os.makedirs(LOCAL_FEATS, exist_ok=True)
have_f = lambda v: os.path.exists(f'{LOCAL_FEATS}/{v}.npz')
FT = sorted({'feats/htstep_feats_%s.tar' % TARS[v].rsplit('_', 1)[1][:-4] for v in VIDEOS})   # cùng số với tar video
FAILED_TXT = {}
for tar in FT:
    want = [v for v in VIDEOS if not have_f(v) and TARS[v].rsplit('_', 1)[1] == tar.rsplit('_', 1)[1]]
    if not want:
        continue
    path = hf_hub_download(HFD['repo'], tar, repo_type='dataset', revision=HFD['revision'], local_dir='/content/hf_feats')
    got = hd.extract_members(path, {f'{v}.npz': f'{LOCAL_FEATS}/{v}.npz' for v in want} | {'failed.txt': f'/content/hf_feats/{tar}.failed.txt'})
    FAILED_TXT[tar] = open(f'/content/hf_feats/{tar}.failed.txt').read().split() if 'failed.txt' in got else []
    os.remove(path)
    print(f'{tar}: {len([g for g in got if g.endswith(".npz")])}/{len(want)} npz | failed.txt: {FAILED_TXT[tar][:10]}')
MISSING = [v for v in VIDEOS if not have_f(v)]
print('thiếu đặc trưng trong htstep-feats/feats:', len(MISSING), MISSING[:10])

EXTRACTED, EXCLUDED = [], {}
if MISSING:   # trích bổ sung bằng đúng bộ trích của repo Athena (tools/extract_internvideo2.py qua athena.encoders.internvideo2)
    from uniav_app.hub import download_encoders
    IV2_REPO = '/content/InternVideo'
    if not os.path.isdir(f'{IV2_REPO}/InternVideo2'):
        subprocess.run(['git', 'clone', '-q', '--depth', '1', 'https://github.com/OpenGVLab/InternVideo', IV2_REPO], check=True)
    print('InternVideo commit:', subprocess.run(['git', '-C', IV2_REPO, 'rev-parse', 'HEAD'], capture_output=True, text=True).stdout.strip())
    download_encoders('/content/ckpt')
    cfg_ex = UConfig.colab(tag='htstep_extract', feature_source='extract', cache_root=H, video_src=VID, video_root='/content/videos',
                           checkpoint=CKPT, expected_run=PR['models']['athena']['run'],
                           iv2_video_encoder='/content/ckpt/internvideo2/InternVideo2-stage2_1b-224p-f4.pt',
                           iv2_audio_encoder='/content/ckpt/internvideo2/audio_6b.pth', iv2_repo=IV2_REPO)
    rec_ex = get_source(cfg_ex).prepare(MISSING, {v: DUR[v] for v in MISSING})   # -> {H}/feats/extract/<id>.npz
    for v, r in rec_ex.items():
        shutil.copy2(r.path, f'{LOCAL_FEATS}/{v}.npz')
        EXTRACTED.append(v)
    EXCLUDED = {v: 'không trích được (xem feats/extract/failed.jsonl)' for v in MISSING if not have_f(v)}
pct_ex = 100 * len(EXCLUDED) / len(VIDEOS)
check('video bị loại vì thiếu đặc trưng <= 10% kho', pct_ex <= PR['data']['features']['stop_if_missing_pct_gt'],
      f'{len(EXCLUDED)} video ({pct_ex:.1f}%): dừng, báo người dùng' if pct_ex > 10 else f'{len(EXCLUDED)} video ({pct_ex:.1f}%)')
KHO = [v for v in VIDEOS if v not in EXCLUDED]
CAPS = [c for c in CAPS if c['video_id'] in set(KHO)]

# định dạng: so với đặc trưng YC2 mà Athena sạch đọc (nguyenminh04/uniav-youcook2-data), 5 video val có cache R6 (dùng lại ở ô 6)
YC2_FINAL = json.load(open(f'{FUSHION}/configs/uemr_yc2_final.json', encoding='utf-8'))
YH8 = YC2_FINAL['code']['uniav_output_hash8']
SUB = {r['video_id']: r for r in json.load(open(f'{O_YC2}/common/subset_videos.json'))['videos']}
R6_YC2 = json.load(open(f'{O_YC2}/common/partitions_{YH8}/uniav_pred.json'))
Y5 = sorted(v for v, r in SUB.items() if r['split'] == 'val')[:5]
DUR_Y5 = {v: float(SUB[v]['duration']) for v in Y5}
cfg_hf = UConfig.colab(tag='yc2_hf', feature_source='hf', cache_root='/content/yc2_feats', checkpoint=CKPT,
                       expected_run=PR['models']['athena']['run'])
rec_hf = get_source(cfg_hf).prepare(Y5, DUR_Y5)
check('5 video YC2 có đặc trưng HF', len(rec_hf) == 5)
YC2_FEATS = str(cfg_hf.feats_dir_for('hf'))

def fmt_stats(folder, vids, dur):
    out = {}
    for v in vids:
        with np.load(f'{folder}/{v}.npz') as z:
            f = hd.feature_format(z)
            n = min(z['v768'].shape[0], z['a768'].shape[0])
            out[v] = {'format': {k: f[k] for k in ('v768', 'v512', 'a768') if k in f}, 'keys': sorted(z.files), 'n': n,
                      'rows/s': n / dur[v], 'audio zero rows %': 100 * float((np.abs(z['a768'][:n].astype(np.float32)).sum(1) == 0).mean()),
                      'v768 row norm': float(np.linalg.norm(z['v768'][:n].astype(np.float32), axis=1).mean())}
    return out
FY, FH = fmt_stats(YC2_FEATS, Y5, DUR_Y5), fmt_stats(LOCAL_FEATS, KHO, DUR)
ref = next(iter(FY.values()))['format']
same = lambda f: set(f) == {'v768', 'v512', 'a768'} and all(f[k][1] == ref[k][1] and f[k][0][1:] == ref[k][0][1:] for k in ref)
bad_fmt = [v for v, s in FH.items() if not same(s['format']) or s['format']['v768'][0][0] != s['format']['v512'][0][0]]
rps = np.array([s['rows/s'] for s in FH.values()])
az = np.array([s['audio zero rows %'] for s in FH.values()])
FEAT = {'YC2 (tham chiếu)': {'format': ref, 'rows/s': [round(s['rows/s'], 3) for s in FY.values()],
                             'v768 row norm': float(np.mean([s['v768 row norm'] for s in FY.values()]))},
        'HT-Step': {'videos': len(FH), 'khác định dạng': bad_fmt[:10], 'rows/s trung vị': float(np.median(rps)),
                    'rows/s min / max': [float(rps.min()), float(rps.max())], '|n - thời lượng| > 2 s': int(sum(abs(s['n'] - DUR[v]) > 2 for v, s in FH.items())),
                    'video audio toàn 0': int((az == 100).sum()), 'audio 0 % trung bình': float(az.mean()),
                    'v768 row norm': float(np.mean([s['v768 row norm'] for s in FH.values()]))},
        'nguồn': {'htstep-feats/feats': len(KHO) - len(EXTRACTED), 'trích bổ sung': EXTRACTED, 'loại': EXCLUDED, 'failed.txt': FAILED_TXT}}
print(json.dumps(FEAT, indent=1, ensure_ascii=False, default=str))
check('định dạng đặc trưng = YC2 (key v768/v512/a768, số chiều, dtype)', not bad_fmt, str(bad_fmt[:5]))
check('1 dòng / giây', 0.95 <= float(np.median(rps)) <= 1.05, f'trung vị {np.median(rps):.3f}')
check('có audio (không phải mọi video audio toàn 0)', (az == 100).mean() < 0.5, f'{int((az == 100).sum())} video audio toàn 0')
INV.update({'features': FEAT, 'kho': len(KHO), 'queries (kho)': len(CAPS)})
wj(f'{RES}/inventory.json', INV)
print(f'KHO: {len(KHO)} video, {len(CAPS)} câu hỏi')
done('A_features', t0, kho=len(KHO), missing=len(MISSING), extracted=len(EXTRACTED), excluded=len(EXCLUDED))

In [ ]:
# ===== 6. Bước B: kiểm tái lập Athena trên 5 video YC2, cắt R6 -> R8, lưu hash =====
from uniav_app.athena_api import load_pipeline
from uniav_app.detect import run_detection
from omni_retrieval.partitions import clamp
from uemr_htstep import partitions as hp
t0 = time.time()
ATH = f'{SEGDIR}/athena961f'
os.makedirs(ATH, exist_ok=True)
KHO_SHA = hashlib.sha256(('\n'.join(KHO) + '\n').encode()).hexdigest()
GT = hd.gt_partition(CAPS)
def write_part(name, part):
    p = f"{SEGDIR}/{hp.FILES[name]}.json"
    wj(p, {v: part[v] for v in sorted(part)})
    return sha256(p)
def clamped(part):
    return {v: clamp(s, DUR[v]) for v, s in part.items()}

SRC_A = rj(f'{ATH}/SOURCE.json')
REUSE = bool(SRC_A) and SRC_A['protocol_sha16'] == PH[:16] and SRC_A['kho_sha256'] == KHO_SHA and SRC_A['athena_sha256'] == ATHENA_SHA
if REUSE or not RUN_ATHENA:
    check('đoạn Athena đã có trên Drive (cùng protocol, kho, checkpoint)', REUSE, f'{ATH}/SOURCE.json')
    R6 = rj(f'{ATH}/r6.json')
    for n in ('r6', 'r8'):
        check(f'hash {n}.json khớp SOURCE.json', sha256(f'{ATH}/{n}.json') == SRC_A['files'][n])
    print('đọc lại đoạn đã cắt, không chạy lại Athena (YC2 tái lập đã kiểm ở lượt cắt):', SRC_A['yc2_repro'])
else:
    common = dict(checkpoint=CKPT, expected_run=PR['models']['athena']['run'], save_z=False, theta=THETA,
                  k_max=PP['k_max'], d_min=PP['d_min'], merge_tiou=PP['merge_tiou'], cache_root='/content/ht_uniav')
    cfg_y = UConfig.colab(tag='yc2_repro', feature_source='drive', drive_feats_src=YC2_FEATS, **common)
    cfg_h = UConfig.colab(tag='htstep', feature_source='drive', drive_feats_src=LOCAL_FEATS, **common)
    for c in (cfg_y, cfg_h):                                     # proposal của lượt này thôi
        if os.path.exists(c.raw_path):
            os.remove(c.raw_path)
    pipe = load_pipeline(cfg_h)                                  # strict=True
    check('cắt bằng state_dict_seg', pipe.seg_model is not pipe.model)
    # 1. tái lập YC2: cùng hàm load (DriveSource -> load_arrays -> detect_features) với HT-Step
    rec_y = get_source(cfg_y).prepare(Y5, DUR_Y5)
    raw_y = run_detection(cfg_y, pipe, rec_y, {v: 'val' for v in Y5}, redo=True)
    r6_y = {v: hp.r6(raw_y[v]['proposals'], DUR_Y5[v], THETA, PP['k_max'], PP['d_min'], PP['merge_tiou']) for v in Y5}
    diff = {v: (r6_y[v], R6_YC2[v]) for v in Y5 if r6_y[v] != R6_YC2[v]}
    check('R6 của 5 video YC2 val trùng khít cache YC2 (partitions_%s/uniav_pred.json)' % YH8, not diff, str(diff)[:500])
    # 2. HT-Step
    rec_h = get_source(cfg_h).prepare(KHO, DUR)
    check('đặc trưng của mọi video trong kho', len(rec_h) == len(KHO), f'{len(rec_h)}/{len(KHO)}')
    raw_h = run_detection(cfg_h, pipe, rec_h, {v: 'val_seen' for v in KHO}, redo=True)
    check('proposal của mọi video', set(raw_h) == set(KHO))
    R6 = {v: hp.r6(raw_h[v]['proposals'], DUR[v], THETA, PP['k_max'], PP['d_min'], PP['merge_tiou']) for v in KHO}
    shutil.copy2(cfg_h.raw_path, f'{ATH}/raw_proposals.jsonl')
    del pipe
    torch.cuda.empty_cache()
PARTS = hp.build(R6, GT, DUR, PAR['R2_L_s'], PAR['d_min'], PAR['L_min_s'], PAR['L_fill_s'])
CL = {n: clamped(p) for n, p in PARTS.items()}
DROP_CLAMP = {LABEL[n]: sum(len(PARTS[n][v]) - len(CL[n][v]) for v in KHO) for n in PARTS}
print('đoạn bị clamp bỏ (bắt đầu trong giây cuối của mp4):', DROP_CLAMP)
PARTS = CL
if REUSE or not RUN_ATHENA:
    check('R8 dựng lại từ R6 = r8.json đã lưu', PARTS['r8'] == rj(f'{ATH}/r8.json'))
else:
    files = {'r6': write_part('uniav_pred', PARTS['uniav_pred']), 'r8': write_part('r8', PARTS['r8'])}
    SRC_A = {'protocol_sha16': PH[:16], 'kho_sha256': KHO_SHA, 'athena_sha256': ATHENA_SHA, 'theta': THETA, 'postprocess': PP,
             'L_min': PAR['L_min_s'], 'L_fill': PAR['L_fill_s'], 'files': files, 'raw_proposals_sha256': sha256(f'{ATH}/raw_proposals.jsonl'),
             'yc2_repro': {'videos': Y5, 'r6_equal': True, 'cache': f'omni_cache/common/partitions_{YH8}/uniav_pred.json'},
             'commit': COMMIT, 'created': time.strftime('%Y-%m-%d %H:%M:%S')}
    wj(f'{ATH}/SOURCE.json', SRC_A)
print('r6.json', SRC_A['files']['r6'][:16], '| r8.json', SRC_A['files']['r8'][:16])
done('B_athena', t0, reuse=REUSE, files=SRC_A['files'])

In [ ]:
# ===== 7. Bước B.3: chất lượng cắt so với GT (mô tả, không có tiêu chí) =====
from omni_retrieval.partitions_uemr import best_tiou, coverage, segment_stats
t0 = time.time()
r6p, gtp = PARTS['uniav_pred'], PARTS['gt']
bt = best_tiou({v: r6p[v] for v in KHO}, {v: gtp[v] for v in KHO})
kp = np.array([len(r6p[v]) for v in KHO]); kg = np.array([len(gtp[v]) for v in KHO])
rank = lambda a: pd.Series(a).rank().to_numpy()
len6 = np.array([te - ts for v in KHO for ts, te in r6p[v]]); leng = np.array([te - ts for v in KHO for ts, te in gtp[v]])
AQ = {'bước GT (clamp, bỏ trùng cửa sổ)': int(len(bt)),
      'recall bước GT': {f'tIoU {t}': 100 * float((bt >= t).mean()) for t in (0.3, 0.5, 0.7)},
      'bước GT bị bỏ sót % (mọi đoạn R6 tIoU < 0.3)': 100 * float((bt < 0.3).mean()),
      'tIoU tốt nhất trung bình': float(bt.mean()),
      'K_pred': pct(kp), 'K_GT': pct(kg), 'K_pred / K_GT trung vị': float(np.median(kp / kg)),
      'tương quan K_pred, K_GT': {'pearson': float(np.corrcoef(kp, kg)[0, 1]), 'spearman': float(np.corrcoef(rank(kp), rank(kg))[0, 1])},
      'độ phủ thời lượng %': {'R6': 100 * float(np.mean([coverage(r6p[v], DUR[v]) for v in KHO])),
                               'GT Full Match': 100 * float(np.mean([coverage(gtp[v], DUR[v]) for v in KHO]))},
      'độ dài đoạn R6 (s)': pct(len6), 'độ dài bước GT (s)': pct(leng),
      'lưu ý': 'GT chỉ gồm bước Full Match: phần video không có nhãn không có nghĩa là không có bước'}
print(json.dumps(AQ, indent=1, ensure_ascii=False))
display(pd.DataFrame([{'partition': LABEL[n], **segment_stats(PARTS[n], {v: DUR[v] for v in KHO})} for n in ROWS]).round(2))
wj(f'{RES}/athena_quality.json', AQ)
done('B3_quality', t0)

In [ ]:
# ===== 8. Bước C: R0, R2, R3, R7; đếm clip; ước lượng thời gian =====
from omni_retrieval.manifest import segment_table, unique_segments, segment_records, caption_records, store_keys, read_store, write_jsonl
t0 = time.time()
FILE_SHA = {n: write_part(n, PARTS[n]) for n in ('global', 'uni_chunk', 'uni_M', 'gt')}
FILE_SHA.update({'uniav_pred': SRC_A['files']['r6'], 'r8': SRC_A['files']['r8']})
for n in ('uniav_pred', 'r8'):
    check(f'{hp.FILES[n]}.json trên Drive = đoạn đang dùng', sha256(f'{SEGDIR}/{hp.FILES[n]}.json') == FILE_SHA[n])
SEG_HASH = hashlib.sha256(json.dumps({hp.FILES[n]: FILE_SHA[n] for n in sorted(FILE_SHA)}, sort_keys=True).encode()).hexdigest()
wj(f'{SEGDIR}/SOURCE.json', {'protocol_sha16': PH[:16], 'kho_sha256': KHO_SHA, 'seg_hash': SEG_HASH,
                             'files': {hp.FILES[n]: FILE_SHA[n] for n in sorted(FILE_SHA)}, 'commit': COMMIT,
                             'created': time.strftime('%Y-%m-%d %H:%M:%S')})
print('SEG_HASH (mọi lượt Omni phải dùng đúng bộ đoạn này):', SEG_HASH)
SEG_ROWS = segment_table(PARTS)
UNIQ = unique_segments(SEG_ROWS)
cnt = pd.DataFrame([{'dòng': LABEL[n], 'video': sum(1 for v in KHO if PARTS[n][v]), 'đoạn': sum(len(PARTS[n][v]) for v in KHO),
                     'vec/video': np.mean([len(PARTS[n][v]) for v in KHO])} for n in ROWS])
display(cnt.round(2))
SEC_REF = 0.313            # s/clip Omni trên A100 (YC2: AZ ô 16, 6.106 clip trong 1.912,9 s); ô 9 đo lại trên 20 clip
print(f'clip khác nhau sau khi gộp seg_key: {len(UNIQ)} | câu hỏi: {len(CAPS)} | ước lượng A100: {len(UNIQ) * SEC_REF / 3600:.2f} giờ / tag, '
      f'{len(UNIQ) * SEC_REF * len(OMNI_TAGS) / 3600:.2f} giờ cho {OMNI_TAGS}')
INV['clips'] = {'unique': len(UNIQ), 'by_row': cnt.to_dict('records'), 'seg_hash': SEG_HASH, 'dropped_by_clamp': DROP_CLAMP}
wj(f'{RES}/inventory.json', INV)
done('C_partitions', t0, unique_clips=len(UNIQ), seg_hash=SEG_HASH)

In [ ]:
# ===== 9. Bước D: Omni encode cho từng tag (kiểm tái lập 5 clip YC2, đo 20 clip, resume) =====
from omni_retrieval.collect import collect_segments, collect_text, local_npz
CKPT_SRC = f'{DRIVE}/Colab Notebooks/code KL/Omni/checkpoint-1948'
FT_ADAPTER, PT_ADAPTER = '/content/my_checkpoint', '/content/adapters/omniretriever-7b'
BASE_MODEL = '/content/WAVE_HOME/WAVE-7B'
BEATS = '/content/WAVE_HOME/BEATs_iter3_plus_AS2M_finetuned_on_AS2M_cpt2.pt'
OM = lambda tag: f'{H}/omni/{tag}'

def omni_env(tag):
    # như uemr_yc2_bo_sung_val.ipynb ô 9: môi trường encode của Omni-fix, encode chạy ở tiến trình con
    from google.colab import userdata
    from safetensors import safe_open
    os.environ['HF_TOKEN'] = userdata.get('HF')
    if not os.path.exists('/content/.fushion_deps_ok'):
        sh(f'pip install -q -r {FUSHION}/requirements.txt')
        open('/content/.fushion_deps_ok', 'w').close()
    sh('pip uninstall -y -q torchao', check=False)
    chk = subprocess.run([sys.executable, '-c', 'import decord, av, peft, torch, transformers; '
                          'print(transformers.__version__, torch.cuda.is_available())'], capture_output=True, text=True)
    last = (chk.stdout.strip().splitlines() or [''])[-1]
    check('môi trường encode (transformers 4.51.3, CUDA)', chk.returncode == 0 and last.split()[:2] == ['4.51.3', 'True'],
          f'{last} {chk.stderr[-300:]}')
    def wave_ok():
        idx = glob.glob(f'{BASE_MODEL}/*.index.json')
        return (os.path.exists(BEATS) and bool(idx) and
                all(os.path.exists(f'{BASE_MODEL}/{s}') for s in set(json.load(open(idx[0]))['weight_map'].values())))
    if not wave_ok():
        sh('hf download nguyenminh04/omni-model --local-dir /content/WAVE_HOME --max-workers 2')
    check('WAVE-7B + BEATs đủ file', wave_ok())
    if tag == 'pt':
        if not os.path.exists(f'{PT_ADAPTER}/adapter_model.safetensors'):
            sh(f"hf download {PR['models']['omni_pt']['name']} --local-dir {PT_ADAPTER} --quiet")
        adapter = PT_ADAPTER
    else:
        if not os.path.exists(f'{FT_ADAPTER}/adapter_model.safetensors'):
            os.makedirs(FT_ADAPTER, exist_ok=True)
            for f in os.listdir(CKPT_SRC):
                if os.path.isfile(f'{CKPT_SRC}/{f}'):
                    shutil.copy2(f'{CKPT_SRC}/{f}', f'{FT_ADAPTER}/{f}')
        adapter = FT_ADAPTER
    with safe_open(f'{adapter}/adapter_model.safetensors', 'pt') as f:
        keys = list(f.keys())
    heads = [h for h in ('classify_linear', 'beats_ln', 'beats_proj') if any(h in k and 'lora_' not in k for k in keys)]
    check(f'{tag}: adapter có fusion head', len(heads) == 3, str(heads))
    info = {'adapter': adapter, 'adapter_sha256': sha256(f'{adapter}/adapter_model.safetensors')}
    print(tag, info)
    return (f'cd {FUSHION} && python -m omni_retrieval.encode --base-model {BASE_MODEL} --beats-path {BEATS} '
            f'--adapter {adapter} --video-root /content/videos'), info

def copy_videos(src_dir, ids):
    os.makedirs('/content/videos', exist_ok=True)
    def one(v):
        src, dst = f'{src_dir}/{v}.mp4', f'/content/videos/{v}.mp4'
        if os.path.exists(dst) and os.path.getsize(dst) == os.path.getsize(src):
            return 0
        shutil.copy2(src, dst + '.part'); os.replace(dst + '.part', dst)
        return 1
    with ThreadPoolExecutor(8) as pool:
        print('đã copy', sum(pool.map(one, list(ids))), 'video về /content/videos')

def encode(modality, manifest, store, log, enc, extra=''):
    # chạy encode; trả (id còn thiếu, tổng id). | tee che mã lỗi của tiến trình con nên tự đếm lại từ store
    with open(log, 'a') as f:
        f.write(f"\n===== {modality} | {time.strftime('%Y-%m-%d %H:%M:%S')} =====\n")
    get_ipython().system(f'{enc} --modality {modality} --manifest "{manifest}" --store "{store}" {extra} 2>&1 | tee -a "{log}"')
    todo = [json.loads(l)['id'] for l in open(manifest) if l.strip()]
    return sorted(set(todo) - store_keys(store, modality)), len(todo)

def failures(store, ids):
    err = {}
    if os.path.exists(f'{store}/failed.jsonl'):
        for l in open(f'{store}/failed.jsonl'):
            if l.strip():
                q = json.loads(l); err[q['id']] = q['error']
    return [{'seg_key': k, 'error': err.get(k, 'không có trong failed.jsonl')} for k in ids]

def yc2_repro(tag, enc):
    # 5 clip R6 của video val YC2 có trong cache YC2 của tag: encode lại, cos >= 0.999
    src = f"{O_YC2}/{tag}/{YH8}"
    if not os.path.exists(f'{src}/segments.npz'):
        print(f'[{tag}] không có cache YC2 {src}/segments.npz: bỏ kiểm tái lập cho tag này (ghi nhật ký)')
        return {'skipped': f'không có {src}/segments.npz'}
    db = str(local_npz(src, f'/content/yc2db_{tag}', ('segments.npz',), ('uniav_pred',)))
    with np.load(f'{db}/segments.npz') as b:
        m = (b['partition'] == 'uniav_pred') & np.isin(b['video_id'], Y5)
        rows = [{'key': k, 'seg_key': s, 'video_id': v, 'ts': float(s.rsplit('__', 1)[1].split('_')[0]),
                 'te': float(s.rsplit('__', 1)[1].split('_')[1])} for k, s, v in zip(b['keys'][m].tolist(), b['seg_key'][m].tolist(),
                                                                                    b['video_id'][m].tolist())][:5]
        ref = {s: e.astype(np.float32) for s, e in zip(b['seg_key'][m].tolist(), b['emb'][m])}
    probe_dir = f'/content/probe_{tag}'
    shutil.rmtree(probe_dir, ignore_errors=True); os.makedirs(probe_dir)
    write_jsonl(f'{probe_dir}/segs.jsonl', segment_records(unique_segments(rows), set()))
    copy_videos(f'{YC2_DATA}/videos', sorted({r['video_id'] for r in rows}))
    left, _ = encode('av', f'{probe_dir}/segs.jsonl', f'{probe_dir}/av', f'{probe_dir}/encode.log', enc, '--num-workers 0')
    check(f'{tag}: encode 5 clip YC2', not left, str(left))
    fresh = read_store(f'{probe_dir}/av', 'av')
    cos = {k: float(fresh[k] @ ref[k] / np.linalg.norm(fresh[k]) / np.linalg.norm(ref[k])) for k in fresh}
    check(f'{tag}: tái lập 5 clip YC2 (cos >= 0.999)', min(cos.values()) >= 0.999, str({k[-20:]: round(c, 5) for k, c in cos.items()}))
    return {'cos': cos}

OMNI_INFO = rj(f'{H}/omni/info.json', {})
if RUN_OMNI:
    caps_rec = [{'caption_id': c['caption_id'], 'sentence': c['sentence']} for c in CAPS]
    for tag in OMNI_TAGS:
        t0 = time.time()
        AV, TX, DBD = f'{OM(tag)}/clips', f'{OM(tag)}/text', f'{OM(tag)}/db'
        log = f'{OM(tag)}/encode.log'
        os.makedirs(OM(tag), exist_ok=True)
        meta = rj(f'{DBD}/meta.json', {})
        av_todo = segment_records(UNIQ, store_keys(AV, 'av'))
        tx_todo = caption_records(caps_rec, store_keys(TX, 'text'))
        if meta.get('seg_hash') == SEG_HASH and meta.get('protocol_sha16') == PH[:16] and not tx_todo and \
                len(av_todo) == len(meta.get('failed', [])):
            print(f'[{tag}] đã encode và gom đủ (seg_hash khớp): bỏ qua'); continue
        check('GPU đủ cho Omni 7B (>= 22 GB: A100 hoặc L4)', GPU_GB >= 22, f'{gpu}, {GPU_GB:.0f} GB: đổi runtime rồi chạy lại')
        enc, info = omni_env(tag)
        info['yc2_repro'] = yc2_repro(tag, enc)
        copy_videos(VID, KHO)
        print(f'[{tag}] clip cần encode: {len(av_todo)} / {len(UNIQ)} | câu hỏi cần encode: {len(tx_todo)}')
        if av_todo:
            write_jsonl(f'{OM(tag)}/segs_todo.jsonl', av_todo)
            n0 = len(store_keys(AV, 'av'))
            if len(av_todo) > 20:   # đo tốc độ trên 20 clip đầu (ghi luôn vào store), rồi ước lượng
                encode('av', f'{OM(tag)}/segs_todo.jsonl', AV, log, enc, '--num-workers 4 --limit 20')
                spc = json.loads(open(f'{AV}/runs.jsonl').read().strip().splitlines()[-1]).get('sec_per_clip') or float('nan')
                info['sec_per_clip_20'] = spc
                print(f'[{tag}] 20 clip: {spc:.3f} s/clip -> còn {len(av_todo) - 20} clip ≈ {(len(av_todo) - 20) * spc / 3600:.2f} giờ')
            left, n = encode('av', f'{OM(tag)}/segs_todo.jsonl', AV, log, enc, '--num-workers 4')
            info['encoded_av'] = len(store_keys(AV, 'av')) - n0
        left = [r['id'] for r in segment_records(UNIQ, store_keys(AV, 'av'))]
        FAIL = failures(AV, left)
        pd.DataFrame(FAIL, columns=['seg_key', 'error']).to_csv(f'{OM(tag)}/failed.csv', index=False)
        fail_pct = 100 * len(FAIL) / len(UNIQ)
        check(f'{tag}: clip lỗi <= 2%', fail_pct <= 2, f'{len(FAIL)} clip ({fail_pct:.2f}%), xem {OM(tag)}/failed.csv')
        if tx_todo:
            write_jsonl(f'{OM(tag)}/caps_todo.jsonl', tx_todo)
            left_t, _ = encode('text', f'{OM(tag)}/caps_todo.jsonl', TX, log, enc, '--batch-size 32')
            check(f'{tag}: encode đủ câu hỏi', not left_t, str(left_t[:5]))
        # gom segments.npz / text.npz ở đĩa local rồi chép lên Drive (np.load trên Drive từng trả file hỏng)
        L = f'/content/htdb_{tag}'
        os.makedirs(L, exist_ok=True)
        seg = collect_segments(SEG_ROWS, AV, f'{L}/segments.npz', allow_missing=True)
        txt = collect_text([{k: c[k] for k in ('caption_id', 'video_id', 'sentence', 'gt_ts', 'gt_te', 'gt_seg_key')} for c in CAPS],
                           TX, f'{L}/text.npz')
        os.makedirs(DBD, exist_ok=True)
        for f in ('segments.npz', 'text.npz'):
            shutil.copy2(f'{L}/{f}', f'{DBD}/{f}')
        meta = {'tag': tag, 'seg_hash': SEG_HASH, 'protocol_sha16': PH[:16], 'segments': seg, 'text': txt,
                'failed': [f['seg_key'] for f in FAIL], **info, 'commit': COMMIT, 'created': time.strftime('%Y-%m-%d %H:%M:%S'),
                'sha256': {f: sha256(f'{L}/{f}') for f in ('segments.npz', 'text.npz')}}
        wj(f'{DBD}/meta.json', meta)
        OMNI_INFO[tag] = meta
        wj(f'{H}/omni/info.json', OMNI_INFO)
        print(f'[{tag}] segments {seg} | text {txt}')
        done(f'D_encode_{tag}', t0, **{k: meta[k] for k in ('segments', 'text', 'adapter_sha256')}, failed=len(FAIL))
        # Omni chạy ở tiến trình con: GPU đã trả khi tiến trình kết thúc; kernel không giữ mô hình nào
        torch.cuda.empty_cache()

In [ ]:
# ===== 10. Bước E: chấm từng tag (strict + multi; mức 1 text -> clip, R0; mức 2 R0-R8, VCMR) =====
from omni_retrieval.evaluate import load_text, paired_bootstrap, rank_metrics, text_to_clip as yc2_text_to_clip, video_ranks
from omni_retrieval.search import SegmentIndex
from uemr_htstep import metrics as hm
MODES = hm.MODES

def boot(a, b, vids, k=1):
    bs = paired_bootstrap(a, b, k, NB, BSEED, 'video', vids)
    return {'A - B': bs['diff'], 'CI': f"[{bs['ci_low']:+.2f}, {bs['ci_high']:+.2f}]", 'ci_low': bs['ci_low'], 'ci_high': bs['ci_high']}

def tag_ready(tag):
    m = rj(f'{OM(tag)}/db/meta.json', {})
    return m.get('seg_hash') == SEG_HASH and m.get('protocol_sha16') == PH[:16]

def ranks_path(tag, row):
    return f'{RES}/ranks_{tag}_{row}.json'

PAIRS = [('r8', 'global'), ('r8', 'uni_M'), ('uniav_pred', 'uni_M'), ('r8', 'uni_chunk'), ('uniav_pred', 'global'),
         ('r8', 'uniav_pred'), ('gt', 'r8')]
VCMR_PAIRS = [('r8', 'uni_M'), ('uniav_pred', 'uni_M'), ('r8', 'uniav_pred')]
if RUN_EVAL:
    for tag in ('pt', 'ft'):
        if not tag_ready(tag):
            print(f'[{tag}] chưa có vector với SEG_HASH này: bỏ qua'); continue
        t0 = time.time()
        DB = str(local_npz(f'{OM(tag)}/db', f'/content/htdb_{tag}', ('segments.npz', 'text.npz'), ROWS))
        text = load_text(DB)
        check(f'{tag}: text.npz cùng thứ tự câu hỏi', [str(x) for x in text['caption_id']] == [c['caption_id'] for c in CAPS])
        vids = text['video_id']
        R, TAB = {}, []
        for p in ROWS:
            idx = SegmentIndex.load(DB, p)
            res = {m: hm.text_to_video(idx, text, CAPS, m) for m in MODES}
            check(f'{tag}/{LABEL[p]}: strict = omni_retrieval.evaluate.video_ranks',
                  np.array_equal(res['strict']['ranks'], video_ranks(idx, text)['ranks']))
            R[p] = res
            for m in MODES:
                TAB.append({'tag': tag, 'mode': m, 'dòng': LABEL[p], **rank_metrics(res[m]['ranks']),
                            'VCMR R@1': float('nan') if p == 'global' else 100 * float(res[m]['vcmr'].mean()),
                            'vec/video': float(np.mean(idx.counts)), 'video có vector': int(idx.n_videos)})
            wj(ranks_path(tag, LABEL[p]), {'caption_id': [c['caption_id'] for c in CAPS], 'video_id': [str(x) for x in vids],
                                           **{m: res[m]['ranks'].tolist() for m in MODES},
                                           **{f'vcmr_{m}': res[m]['vcmr'].astype(int).tolist() for m in MODES}})
        # mức 1: text -> clip trên mọi clip GT của kho
        ck, ce = hm.gt_clips(f'{DB}/segments.npz')
        T2C = {m: hm.text_to_clip(text, CAPS, ck, ce, m) for m in MODES}
        try:
            ref = yc2_text_to_clip(DB)['subset']
            check(f'{tag}: text -> clip strict = omni_retrieval.evaluate.text_to_clip', all(abs(rank_metrics(T2C['strict'])[k] - ref[k]) < 1e-9
                                                                                         for k in ('R@1', 'R@5', 'R@10', 'MedR')))
        except Exception as e:   # noqa: BLE001 - clip GT lỗi encode: text_to_clip của YC2 không chạy được
            print(f'[{tag}] không đối chiếu được text_to_clip của YC2: {e}')
        for m in MODES:
            TAB.append({'tag': tag, 'mode': m, 'dòng': 'text -> clip (R7 clip)', **rank_metrics(T2C[m]), 'VCMR R@1': float('nan'),
                        'vec/video': float('nan'), 'video có vector': len(ck)})
        wj(ranks_path(tag, 't2c'), {'caption_id': [c['caption_id'] for c in CAPS], 'video_id': [str(x) for x in vids],
                                    **{m: T2C[m].tolist() for m in MODES}})
        PB = []
        for m in MODES:
            for a, b in PAIRS:
                PB.append({'tag': tag, 'mode': m, 'chỉ số': 'R@1', 'cặp': f'{LABEL[a]} - {LABEL[b]}',
                           **boot(R[a][m]['ranks'], R[b][m]['ranks'], vids)})
            for a, b in VCMR_PAIRS:
                PB.append({'tag': tag, 'mode': m, 'chỉ số': 'VCMR R@1', 'cặp': f'{LABEL[a]} - {LABEL[b]}',
                           **boot(np.where(R[a][m]['vcmr'], 1, 2), np.where(R[b][m]['vcmr'], 1, 2), vids)})
        tab = pd.DataFrame(TAB)
        print(f'=== {tag}: multi-positive (chính) ==='); display(tab[tab['mode'] == 'multi'].drop(columns=['tag', 'mode']).round(2))
        print(f'=== {tag}: strict ==='); display(tab[tab['mode'] == 'strict'].drop(columns=['tag', 'mode']).round(2))
        pb = pd.DataFrame(PB)
        display(pb.drop(columns=['ci_low', 'ci_high', 'tag']).round(2))
        wj(f'{RES}/metrics_{tag}.json', {'tag': tag, 'seg_hash': SEG_HASH, 'protocol_sha16': PH[:16], 'queries': len(CAPS),
                                         'videos': len(KHO), 'omni': {k: OMNI_INFO.get(tag, {}).get(k) for k in ('adapter', 'adapter_sha256', 'failed')},
                                         'table': TAB, 'pairs': PB, 'bootstrap': EV['bootstrap'], 'commit': COMMIT})
        done(f'E_eval_{tag}', t0)

In [ ]:
# ===== 11. Tiêu chí mục 6 -> criteria.json =====
t0 = time.time()
def pair_of(tag, cap, mode='multi', what='R@1'):
    m = rj(f'{RES}/metrics_{tag}.json')
    if m is None or m['seg_hash'] != SEG_HASH:
        return None
    return next(p for p in m['pairs'] if p['mode'] == mode and p['chỉ số'] == what and p['cặp'] == cap)
def rk(tag, row, mode='multi'):
    j = rj(ranks_path(tag, row))
    return None if j is None else np.array(j[mode])
CRIT = {'protocol_sha16': PH[:16], 'seg_hash': SEG_HASH, 'main_cell': CR['main_cell']}
T = {}
for name, cap in (('T1', 'R8 - R0'), ('T2', 'R8 - R3'), ('T3', 'R6 - R3'), ('T5', 'R8 - R2')):
    p = pair_of('ft', cap)
    T[name] = None if p is None else {'cặp': cap, 'A - B': p['A - B'], 'CI': p['CI'], 'ci_low': p['ci_low'],
                                      'đạt': None if name == 'T5' else bool(p['ci_low'] > 0)}
pt1, pt2 = pair_of('pt', 'R8 - R0'), pair_of('pt', 'R8 - R3')
T['T4'] = None if pt1 is None else {'T1 pt': {'A - B': pt1['A - B'], 'CI': pt1['CI'], 'đạt': bool(pt1['ci_low'] > 0)},
                                    'T2 pt': {'A - B': pt2['A - B'], 'CI': pt2['CI'], 'đạt': bool(pt2['ci_low'] > 0)}}
def direction(b):
    s = CR['T6']['sentence']
    return s['ci_low > 0'] if b['ci_low'] > 0 else s['ci_high < 0'] if b['ci_high'] < 0 else s['ci contains 0']
T6 = {}
for row in ('R8', 't2c'):
    f, p = rk('ft', row), rk('pt', row)
    if f is None or p is None:
        continue
    vids = rj(ranks_path('ft', row))['video_id']
    check(f'T6 {row}: cùng câu hỏi ở hai tag', rj(ranks_path('ft', row))['caption_id'] == rj(ranks_path('pt', row))['caption_id'])
    b = boot(f, p, vids)
    T6['R8 R@1 text -> video' if row == 'R8' else 'text -> clip R@1'] = {**b, 'hướng': direction(b)}
T['T6'] = T6 or None
for k in ('T1', 'T2', 'T3', 'T4', 'T5', 'T6'):
    T[k] = {'test': CR[k]['test'], 'kết quả': T[k], 'câu nếu đạt': CR[k]['sentence']}
CRIT['criteria'] = T
if T['T1']['kết quả'] is None:
    CRIT['conclusion'] = 'chưa có kết quả Omni ft (ô chính): chạy ô 9-10 với tag ft'
else:
    t1, t2 = T['T1']['kết quả']['đạt'], T['T2']['kết quả']['đạt']
    C = CR['conclusion']
    CRIT['conclusion'] = C[0]['text'] if t1 and t2 else C[1]['text'] if t1 else C[2]['text']
missing = [t for t in ('pt', 'ft') if rj(f'{RES}/metrics_{t}.json') is None]
CRIT['missing_tags'] = missing
print(json.dumps(CRIT, indent=1, ensure_ascii=False))
if missing:
    print('THIẾU kết quả tag:', missing, '-> T4 / T6 chưa tính' if 'pt' in missing else '')
wj(f'{RES}/criteria.json', CRIT)
done('criteria', t0, conclusion=CRIT['conclusion'], missing=missing)

In [ ]:
# ===== 12. Bước F: so sánh pt với ft (cần đủ hai tag, cùng SEG_HASH) =====
t0 = time.time()
need = [t for t in ('pt', 'ft') if not (rj(f'{RES}/metrics_{t}.json') or {}).get('seg_hash') == SEG_HASH]
if need:
    print('CHƯA so sánh được: thiếu kết quả tag', need, f'(cùng SEG_HASH {SEG_HASH[:16]}). Chạy ô 9-10 với OMNI_TAGS = {need}.')
else:
    ITEMS = [('text -> clip R@1', 't2c', None), ('R@1 R0', 'R0', None), ('R@1 R3', 'R3', None), ('R@1 R6', 'R6', None),
             ('R@1 R8', 'R8', None), ('VCMR R@1 R6', 'R6', 'vcmr'), ('VCMR R@1 R8', 'R8', 'vcmr')]
    CMP = []
    for m in MODES:
        for name, row, kind in ITEMS:
            jf, jp = rj(ranks_path('ft', row)), rj(ranks_path('pt', row))
            check(f'{name}: cùng câu hỏi', jf['caption_id'] == jp['caption_id'])
            if kind:
                f, p = np.where(np.array(jf[f'vcmr_{m}']) == 1, 1, 2), np.where(np.array(jp[f'vcmr_{m}']) == 1, 1, 2)
            else:
                f, p = np.array(jf[m]), np.array(jp[m])
            b = boot(f, p, jf['video_id'])
            CMP.append({'mode': m, 'chỉ số': name, 'pt': 100 * float((p <= 1).mean()), 'ft': 100 * float((f <= 1).mean()),
                        'ft - pt': b['A - B'], 'CI': b['CI'], 'ci_low': b['ci_low'], 'ci_high': b['ci_high']})
        for cap in ('R8 - R0', 'R8 - R3'):
            q = {t: next(x for x in rj(f'{RES}/metrics_{t}.json')['pairs'] if x['mode'] == m and x['chỉ số'] == 'R@1' and x['cặp'] == cap)
                 for t in ('pt', 'ft')}
            CMP.append({'mode': m, 'chỉ số': cap + ' (trong từng tag)', 'pt': q['pt']['A - B'], 'ft': q['ft']['A - B'],
                        'ft - pt': float('nan'), 'CI': f"pt {q['pt']['CI']} | ft {q['ft']['CI']}"})
    cmp_ = pd.DataFrame(CMP)
    print('=== multi-positive (chính) ==='); display(cmp_[cmp_['mode'] == 'multi'].drop(columns=['mode', 'ci_low', 'ci_high']).round(2))
    print('=== strict ==='); display(cmp_[cmp_['mode'] == 'strict'].drop(columns=['mode', 'ci_low', 'ci_high']).round(2))
    wj(f'{RES}/compare_pt_ft.json', {'seg_hash': SEG_HASH, 'protocol_sha16': PH[:16], 'rows': CMP, 'bootstrap': EV['bootstrap']})
    done('F_compare', t0)

In [ ]:
# ===== 13. Xuất kết quả, tóm tắt =====
import zipfile
files = sorted(glob.glob(f'{RES}/*.json'))
ZIP = f'/content/uemr_htstep_0shot_{PROTO_SHA16[:8]}.zip'
with zipfile.ZipFile(ZIP, 'w', zipfile.ZIP_DEFLATED) as z:
    for f in files + [f'{SEGDIR}/SOURCE.json', f'{SEGDIR}/athena961f/SOURCE.json', f'{H}/htstep_val_manifest.json', MAN_PATH]:
        if os.path.exists(f):
            z.write(f, os.path.relpath(f, H))
print('kết quả trên Drive:', RES)
for f in files:
    print(f'  {os.path.basename(f):32s} {os.path.getsize(f):>10d} B  sha256 {sha256(f)[:16]}')
print('zip để tải về (chép vào Fushion results/uemr_htstep_0shot/%s/):' % PROTO_SHA16[:8], ZIP)
SUM = {'protocol': PH[:16], 'commit': COMMIT, 'seg_hash': SEG_HASH, 'kho': len(KHO), 'câu hỏi': len(CAPS),
       'tags có kết quả': [t for t in ('pt', 'ft') if os.path.exists(f'{RES}/metrics_{t}.json')],
       'kết luận': (rj(f'{RES}/criteria.json') or {}).get('conclusion'), 'thời gian phiên (phút)': round((time.time() - T_START) / 60, 1)}
print(json.dumps(SUM, indent=1, ensure_ascii=False))
done('export', time.time(), **SUM)
try:
    from google.colab import files as _f
    _f.download(ZIP)
except Exception as e:   # noqa: BLE001
    print('tải zip thủ công:', e)